In [1]:
import subprocess

r = subprocess.run(
    ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq zstd"],
    capture_output=True,
    text=True
)

print(r.stdout)
if r.stderr:
    print(r.stderr)

print("zstd:", subprocess.run(
    ["bash", "-lc", "which zstd"],
    capture_output=True,
    text=True
).stdout.strip())

Selecting previously unselected package zstd.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to p

In [2]:
import subprocess

r = subprocess.run(
    ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
    capture_output=True,
    text=True
)

print(r.stdout)
if r.stderr:
    print(r.stderr)

print("\n=== VERIFY ===")
subprocess.run(["bash", "-lc", "ollama --version"])

KeyboardInterrupt: 

In [ ]:
import subprocess
import time
import urllib.request

print("=== STARTING OLLAMA ===")

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    with urllib.request.urlopen("http://127.0.0.1:11434/", timeout=10) as r:
        print("HTTP:", r.status)
        print("Response:", r.read().decode())
except Exception as e:
    print("OLLAMA FAILED:", e)

print("\n=== VERSION ===")
subprocess.run(["ollama", "--version"])

In [ ]:
import subprocess

print("=== AVAILABLE MODELS ===")
subprocess.run(["ollama", "list"])

In [ ]:
import subprocess

print("=== PULLING QWEN 2.5 CODER 32B ===")
r = subprocess.run(
    ["ollama", "pull", "qwen2.5-coder:32b"],
    text=True
)

print("\nExit code:", r.returncode)

In [ ]:
import subprocess
import time
import urllib.request

print("=== RESTARTING OLLAMA ===")

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    with urllib.request.urlopen("http://127.0.0.1:11434/", timeout=10) as r:
        print("Server:", r.status, r.read().decode())
except Exception as e:
    print("Server FAILED:", e)

print("\n=== MODEL LIST ===")
subprocess.run(["ollama", "list"])

In [ ]:
import subprocess

print("=== RESTORING QWEN 2.5 CODER 32B ===")
r = subprocess.run(
    ["ollama", "pull", "qwen2.5-coder:32b"],
    text=True
)

print("\nExit code:", r.returncode)

In [ ]:
import subprocess
subprocess.run(["ollama", "list"])

In [ ]:
import json, time, uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import Request, urlopen

MODEL = "qwen2.5-coder:32b"
OLLAMA_API = "http://127.0.0.1:11434"
WORKER_HOST = "127.0.0.1"
WORKER_PORT = 8787
START_TIME = time.time()

def ollama_health():
    try:
        req = Request(f"{OLLAMA_API}/", method="GET")
        with urlopen(req, timeout=5) as response:
            body = response.read().decode()
        return response.status == 200 and "Ollama is running" in body
    except Exception:
        return False

def ollama_generate(prompt, timeout=300):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 128
        }
    }
    body = json.dumps(payload).encode()
    request = Request(
        f"{OLLAMA_API}/api/generate",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urlopen(request, timeout=timeout) as response:
        return response.status, json.loads(response.read().decode())

class VajraWorkerHandler(BaseHTTPRequestHandler):
    def _send_json(self, status, payload):
        body = json.dumps(payload, sort_keys=True).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _read_json(self):
        length = int(self.headers.get("Content-Length", "0"))
        return json.loads(self.rfile.read(length).decode())

    def do_GET(self):
        if self.path == "/health":
            healthy = ollama_health()
            self._send_json(200 if healthy else 503, {
                "status": "ok" if healthy else "unhealthy",
                "worker": "vajra-kaggle-worker",
                "runtime": "ollama",
                "model": MODEL,
                "uptime_seconds": round(time.time() - START_TIME, 2)
            })
            return

        if self.path == "/capabilities":
            self._send_json(200, {
                "worker": "vajra-kaggle-worker",
                "protocol": "vajra-worker-v1",
                "runtime": "ollama",
                "model": MODEL,
                "capabilities": ["completion", "tools", "insert"],
                "context_length": 32768,
                "quantization": "Q4_K_M",
                "parameter_size": "32.8B",
                "timeout_seconds": 300
            })
            return

        self._send_json(404, {"error": "not_found"})

    def do_POST(self):
        if self.path != "/infer":
            self._send_json(404, {"error": "not_found"})
            return

        request_id = str(uuid.uuid4())
        started = time.time()

        try:
            payload = self._read_json()
            prompt = payload.get("prompt")

            if not isinstance(prompt, str) or not prompt.strip():
                self._send_json(400, {
                    "request_id": request_id,
                    "error": "prompt_required"
                })
                return

            timeout = int(payload.get("timeout_seconds", 300))
            status, result = ollama_generate(prompt, timeout=timeout)
            elapsed = time.time() - started

            self._send_json(200, {
                "request_id": request_id,
                "worker": "vajra-kaggle-worker",
                "model": MODEL,
                "status": "completed",
                "elapsed_seconds": round(elapsed, 3),
                "response": result.get("response", ""),
                "usage": {
                    "total_duration_ns": result.get("total_duration"),
                    "load_duration_ns": result.get("load_duration"),
                    "prompt_eval_count": result.get("prompt_eval_count"),
                    "eval_count": result.get("eval_count")
                }
            })

        except Exception as exc:
            self._send_json(502, {
                "request_id": request_id,
                "worker": "vajra-kaggle-worker",
                "status": "failed",
                "elapsed_seconds": round(time.time() - started, 3),
                "error_type": type(exc).__name__,
                "error": str(exc)
            })

    def log_message(self, format, *args):
        pass

print("=== STARTING VAJRA WORKER ===")
print("Ollama healthy:", ollama_health())

server = ThreadingHTTPServer((WORKER_HOST, WORKER_PORT), VajraWorkerHandler)
print(f"Worker listening on http://{WORKER_HOST}:{WORKER_PORT}")
server.serve_forever()

In [ ]:
import subprocess

print("=== INSTALLING CLOUDFLARED ===")

subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True,
    check=True
)

subprocess.run(["cloudflared", "--version"], check=True)

In [ ]:
import subprocess, time, re

print("=== STARTING CLOUDFLARE QUICK TUNNEL ===")

p = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8787"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(30):
    line = p.stdout.readline()
    if line:
        print(line.strip())
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\n=== TUNNEL URL ===")
            print(match.group(0))
            break
    time.sleep(1)

In [ ]:
import subprocess
import time

print("=== STARTING OLLAMA IN BACKGROUND ===")

ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

check = subprocess.run(
    "curl -sS --max-time 10 http://127.0.0.1:11434/",
    shell=True,
    capture_output=True,
    text=True
)

print("Ollama PID:", ollama.pid)
print("HTTP:", check.stdout.strip() if check.stdout else check.stderr.strip())

In [ ]:
import subprocess
import time
import os
import signal

print("=== CLEAN WORKER RESET ===")

# Kill anything currently using port 8787.
subprocess.run(
    "fuser -k 8787/tcp 2>/dev/null || true",
    shell=True
)

time.sleep(2)

# Confirm port is free.
port_check = subprocess.run(
    "ss -ltnp | grep ':8787' || true",
    shell=True,
    capture_output=True,
    text=True
)

print("Port state:", port_check.stdout.strip() or "FREE")

# Write a minimal worker to disk.
worker_code = r'''
import json
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import Request, urlopen

HOST = "127.0.0.1"
PORT = 8787
OLLAMA = "http://127.0.0.1:11434"
MODEL = "qwen2.5-coder:32b"

class Handler(BaseHTTPRequestHandler):

    def send_json(self, code, data):
        body = json.dumps(data).encode()

        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()

        self.wfile.write(body)

    def do_GET(self):

        if self.path == "/health":
            try:
                req = Request(OLLAMA, method="GET")

                with urlopen(req, timeout=3) as r:
                    ok = (
                        r.status == 200
                        and r.read().decode().strip()
                        == "Ollama is running"
                    )

                self.send_json(
                    200 if ok else 503,
                    {
                        "status": "ok" if ok else "unhealthy",
                        "worker": "vajra-kaggle-worker",
                        "model": MODEL
                    }
                )

            except Exception as e:
                self.send_json(
                    503,
                    {
                        "status": "unhealthy",
                        "worker": "vajra-kaggle-worker",
                        "error": str(e)
                    }
                )

            return

        if self.path == "/capabilities":
            self.send_json(
                200,
                {
                    "worker": "vajra-kaggle-worker",
                    "protocol": "vajra-worker-v1",
                    "runtime": "ollama",
                    "model": MODEL,
                    "capabilities": [
                        "completion",
                        "tools",
                        "insert"
                    ],
                    "context_length": 32768
                }
            )
            return

        self.send_json(404, {"error": "not_found"})

    def do_POST(self):

        if self.path != "/infer":
            self.send_json(404, {"error": "not_found"})
            return

        try:
            length = int(self.headers.get("Content-Length", "0"))
            payload = json.loads(self.rfile.read(length).decode())

            prompt = payload.get("prompt")

            if not isinstance(prompt, str) or not prompt.strip():
                self.send_json(
                    400,
                    {"error": "prompt_required"}
                )
                return

            request_id = str(uuid.uuid4())
            started = time.time()

            ollama_payload = json.dumps({
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "num_predict": 128
                }
            }).encode()

            req = Request(
                f"{OLLAMA}/api/generate",
                data=ollama_payload,
                headers={
                    "Content-Type": "application/json"
                },
                method="POST"
            )

            with urlopen(
                req,
                timeout=int(
                    payload.get("timeout_seconds", 300)
                )
            ) as response:

                result = json.loads(
                    response.read().decode()
                )

            self.send_json(
                200,
                {
                    "request_id": request_id,
                    "worker": "vajra-kaggle-worker",
                    "model": MODEL,
                    "status": "completed",
                    "elapsed_seconds": round(
                        time.time() - started,
                        3
                    ),
                    "response": result.get(
                        "response",
                        ""
                    ),
                    "usage": {
                        "total_duration_ns":
                            result.get("total_duration"),
                        "load_duration_ns":
                            result.get("load_duration"),
                        "prompt_eval_count":
                            result.get("prompt_eval_count"),
                        "eval_count":
                            result.get("eval_count")
                    }
                }
            )

        except Exception as e:

            self.send_json(
                502,
                {
                    "worker": "vajra-kaggle-worker",
                    "status": "failed",
                    "error_type": type(e).__name__,
                    "error": str(e)
                }
            )

    def log_message(self, *args):
        pass


server = ThreadingHTTPServer(
    (HOST, PORT),
    Handler
)

server.daemon_threads = True
server.serve_forever()
'''

with open("/tmp/vajra_worker.py", "w") as f:
    f.write(worker_code)

log = open("/tmp/vajra_worker.log", "w")

worker = subprocess.Popen(
    ["python3", "/tmp/vajra_worker.py"],
    stdout=log,
    stderr=log,
    start_new_session=True
)

with open("/tmp/vajra_worker.pid", "w") as f:
    f.write(str(worker.pid))

print("Worker PID:", worker.pid)

# Give server time to bind.
time.sleep(2)

health = subprocess.run(
    [
        "curl",
        "-sS",
        "--max-time",
        "5",
        "http://127.0.0.1:8787/health"
    ],
    capture_output=True,
    text=True
)

print("Health:", health.stdout.strip() or health.stderr.strip())

print("\n=== WORKER LOG ===")
subprocess.run(
    ["tail", "-20", "/tmp/vajra_worker.log"]
)·